# 01. DATA INTEGRATION - THU THẬP VÀ TÍCH HỢP DỮ LIỆU
 Data Preparation (Chuẩn bị dữ liệu)

**Mục tiêu:** 
1. Tải tập dữ liệu gốc từ Kaggle.
2. Hợp nhất dữ liệu lịch sử (2009-2023) và dữ liệu mới (2025).
3. Làm giàu dữ liệu bằng cách truy vấn 12 thuộc tính âm học (Audio Features) từ Spotify API.

In [ ]:
import pandas as pd
import kagglehub
import os
import shutil
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import time
import warnings
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
print("--- Thư viện đã sẵn sàng ---")

## 1. Tải dữ liệu từ Kaggle

In [ ]:
# Tải dataset
path = kagglehub.dataset_download("wardabilal/spotify-global-music-dataset-20092025")
print(f"Dataset đã tải về tại: {path}")

# Di chuyển vào thư mục dự án data/raw
raw_folder = "../data/raw/"
os.makedirs(raw_folder, exist_ok=True)

for file in os.listdir(path):
    if file.endswith(".csv"):
        shutil.copy(os.path.join(path, file), raw_folder)
        print(f"Đã sao chép: {file}")

## 2. Hợp nhất các tập dữ liệu gốc

In [ ]:
# Đọc dữ liệu
df_2025 = pd.read_csv('../data/raw/spotify_data clean.csv')
df_history = pd.read_csv('../data/raw/track_data_final.csv')

# Gộp thô
df_merged = pd.concat([df_2025, df_history], ignore_index=True)

# Lọc các track_id duy nhất để tối ưu hóa việc gọi API
unique_tracks = df_merged.drop_duplicates(subset=['track_id'])
print(f"Số lượng bản ghi sau khi gộp: {len(df_merged)}")
print(f"Số lượng ID duy nhất cần truy vấn API: {len(unique_tracks)}")

## 3. Truy vấn Spotify API (Audio Features)

In [ ]:
# Cấu hình xác thực (Thay thế bằng Client ID của bạn)
# Load các biến từ file .env
load_dotenv() 

client_id = os.getenv('SPOTIPY_CLIENT_ID')
client_secret = os.getenv('SPOTIPY_CLIENT_SECRET')

# Khởi tạo Spotipy
auth_manager = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret)
sp = spotipy.Spotify(auth_manager=auth_manager)

track_ids = unique_tracks['track_id'].tolist()
audio_features = []

# Vòng lặp lấy dữ liệu theo đợt
for i in range(0, len(track_ids), 100):
    batch = track_ids[i:i+100]
    try:
        results = sp.audio_features(batch)
        audio_features.extend([res for res in results if res is not None])
        if (i % 1000 == 0): print(f"Đã xử lý: {i}/{len(track_ids)}")
    except Exception as e:
        print(f"Lỗi tại batch {i}: {e}")
        time.sleep(5)

df_features = pd.DataFrame(audio_features)

## 4. Hoàn thiện tập dữ liệu tích hợp

In [ ]:
# Merge dữ liệu
final_df = pd.merge(df_merged, df_features, left_on='track_id', right_on='id', how='inner')

# Xóa các cột định danh dư thừa từ API
cols_to_drop = ['id', 'uri', 'track_href', 'analysis_url', 'type']
final_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

# Lưu file
output_path = '../data/processed/merged_data.csv'
final_df.to_csv(output_path, index=False)

print(f"--- HOÀN THÀNH TÍCH HỢP ---")
print(f"Tổng số thuộc tính hiện có: {final_df.shape[1]}")
print(f"File đã lưu tại: {output_path}")